## 1. Bibliotecas

In [ ]:
import os, sys, shutil

In [ ]:
IN_COLAB = 'google.colab' in sys.modules

In [ ]:
if IN_COLAB:
  !pip -q install lime statsmodels python-dotenv
  !python -m spacy download pt_core_news_sm

In [ ]:
import json
import time
import re
from pathlib import Path
import numpy as np
import pandas as pd
import spacy
import torch
import statsmodels.api as sm
from lime.lime_text import LimeTextExplainer
from scipy.stats import chi2_contingency, mannwhitneyu, wilcoxon
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer

## 2. Configuração do ambiente

In [ ]:
if IN_COLAB:
  REPO_URL = 'https://github.com/httpsemilly/xplica.ai.git'
  REPO_DIR = Path('/content/xplica-ai')

  if not REPO_DIR.exists():
    !git clone --filter=blob:none --sparse {REPO_URL} {REPO_DIR}
    !git -C {REPO_DIR} sparse-checkout set data

  os.chdir(REPO_DIR)

  !pip -q install lime statsmodels python-dotenv
  !python -m spacy download pt_core_news_sm

  from google.colab import drive
  drive.mount('/content/drive', force_remount=False)
  DRIVE_ROOT = Path('/content/drive/MyDrive/xplica')

  for sub in ('models', 'results'):
    local = REPO_DIR / sub
    drive_dir = DRIVE_ROOT / sub
    drive_dir.mkdir(parents=True, exist_ok=True)

    if not local.exists():
      local.symlink_to(drive_dir)

In [ ]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data').exists())

MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'
MAX_LENGTH = 512
label_map = {'fake': 0, 'true': 1}

MODELS_DIR = ROOT / 'models'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
HF_USER = os.environ.get('XPLICA_HF_USER')

SEED = 42
SAMPLE_SIZE = 40
LIME_NUM_SAMPLES = 1000
LIME_NUM_FEATURES = 10
SCENARIO = 'scenario1'
WEEKDAYS = re.compile(r'\b(?:segunda|terça|quarta|quinta|sexta)-feira\b|\bsábado\b|\bdomingo\b', re.IGNORECASE)

try:
  nlp = spacy.load('pt_core_news_sm')
except OSError:
  spacy.cli.download('pt_core_news_sm')
  nlp = spacy.load('pt_core_news_sm')

STOP = nlp.Defaults.stop_words
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 3. Carregamento dos dados e tokenizer

In [ ]:
df_news = pd.read_csv(ROOT / 'data' / 'processed' / 'news_dataset.csv')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)

## 4. Funções auxiliares

In [ ]:
def load_scenario_model(name):
  local_path = MODELS_DIR / f'bertimbau-{name}'
  if local_path.exists():
    print(f'Carregando {name} do disco local: {local_path}')
    return AutoModelForSequenceClassification.from_pretrained(str(local_path)).to(device).eval()

  print(f'{name} não encontrado localmente, tentando HF Hub...')
  return AutoModelForSequenceClassification.from_pretrained(hub_repo(name)).to(device).eval()

def make_predict_proba(model, batch_size=32):
  def predict_proba(texts):
    texts = list(texts)
    all_probs = []
    for i in range(0, len(texts), batch_size):
      inputs = tokenizer(texts[i:i + batch_size], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(device)
      with torch.no_grad():
        logits = model(**inputs).logits
      all_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())

    return np.vstack(all_probs)

  return predict_proba

def filter_test_data(df, fake_origin, test_split='test'):
  df_true = df[(df['label'] == 'true') & (df['split'] == test_split)].copy()
  df_fake = df[(df['label'] == 'fake') & (df['origin'] == fake_origin) & (df['split'] == test_split)].copy()
  df_test = pd.concat([df_true, df_fake]).copy()
  df_test['label_id'] = df_test['label'].map(label_map)

  return df_test

## 5. Erros do cenário 2 e amostras

In [ ]:
model_s1 = load_scenario_model('scenario1')
predict_proba_s1 = make_predict_proba(model_s1)

df_test_s2 = filter_test_data(df_news, fake_origin='synthetic').reset_index(drop=True)
df_test_s2['predicted_label_id'] = predict_proba_s1(df_test_s2['text']).argmax(axis=1)

false_positives = df_test_s2[(df_test_s2['label_id'] == 0) & (df_test_s2['predicted_label_id'] == 1)]
true_negatives = df_test_s2[(df_test_s2['label_id'] == 0) & (df_test_s2['predicted_label_id'] == 0)]

print(f'False positives: {len(false_positives)}')
print(f'True negatives: {len(true_negatives)}')

fp_sample = false_positives.sample(n=SAMPLE_SIZE, random_state=SEED)
tn_sample = true_negatives.sample(n=SAMPLE_SIZE, random_state=SEED)

## 6. LIME

In [ ]:
explainer = LimeTextExplainer(class_names=['fake', 'true'], random_state=SEED)

def explain_samples(df_sample, group_name):
  rows = []
  for _, row in df_sample.iterrows():
    exp = explainer.explain_instance(row['text'], predict_proba_s1, labels=(1,), num_features=LIME_NUM_FEATURES, num_samples=LIME_NUM_SAMPLES)
    for word, weight in exp.as_list(label=1):
      rows.append({'id': row['id'], 'group': group_name, 'word': str(word), 'weight': weight})

  return pd.DataFrame(rows)

def load_or_compute(path, df_sample, group_name):
  if path.exists():
    return pd.read_csv(path, encoding='utf-8-sig')
  df = explain_samples(df_sample, group_name)
  df.to_csv(path, index=False, encoding='utf-8-sig')

  return df

In [ ]:
lime_fp = load_or_compute(RESULTS_DIR / 'lime_scenario2_fp.csv', fp_sample, 'FP')
lime_tn = load_or_compute(RESULTS_DIR / 'lime_scenario2_tn.csv', tn_sample, 'TN')

print(lime_fp.shape, lime_tn.shape)

## 7. Palavras de maior peso

In [ ]:
stopwords_pt = nlp.Defaults.stop_words

def top_words(df, n=15, min_count=2, remove_stopwords=False):
  if remove_stopwords:
    df = df[~df['word'].str.lower().isin(stopwords_pt)]
  agg = df.groupby('word')['weight'].agg(['mean', 'count'])
  return agg[agg['count'] >= min_count].sort_values('mean', ascending=False).head(n)

In [ ]:
for group, df in [('FP', lime_fp), ('TN', lime_tn)]:
  print(f'--- {group}: words pushing toward "true" ---')
  print(top_words(df))
  print(f'--- {group} (without stop words) ---')
  print(top_words(df, remove_stopwords=True))

## 8. spaCy

In [ ]:
def extract_linguistic_features(text):
  doc = nlp(text)

  total_tokens = len([t for t in doc if not t.is_punct and not t.is_space])
  if total_tokens == 0:
    return None

  pos_counts = {pos: sum(1 for t in doc if t.pos_ == pos) for pos in ['PROPN', 'NOUN', 'ADJ', 'VERB', 'ADV']}

  ent_counts = {'PER': 0, 'ORG': 0, 'LOC': 0, 'MISC': 0}
  for ent in doc.ents:
    if ent.label_ in ent_counts:
      ent_counts[ent.label_] += 1

  sentences = list(doc.sents)
  avg_sentence_length = total_tokens / len(sentences) if sentences else 0
  denominator = pos_counts['NOUN'] + pos_counts['VERB']
  emotiveness = (pos_counts['ADJ'] + pos_counts['ADV']) / denominator if denominator > 0 else 0

  return {
    'propn_ratio': pos_counts['PROPN'] / total_tokens,
    'noun_ratio': pos_counts['NOUN'] / total_tokens,
    'adj_ratio': pos_counts['ADJ'] / total_tokens,
    'verb_ratio': pos_counts['VERB'] / total_tokens,
    'adv_ratio': pos_counts['ADV'] / total_tokens,
    'per_ratio': ent_counts['PER'] / total_tokens,
    'org_ratio': ent_counts['ORG'] / total_tokens,
    'loc_ratio': ent_counts['LOC'] / total_tokens,
    'misc_ratio': ent_counts['MISC'] / total_tokens,
    'avg_sentence_length': avg_sentence_length,
    'emotiveness': emotiveness,
    'total_tokens': total_tokens
  }

def analyze_group(df_sample, group_name):
  rows = []
  for _, row in df_sample.iterrows():
    features = extract_linguistic_features(row['text'])
    if features:
      features['id'] = row['id']
      features['group'] = group_name
      rows.append(features)

  return pd.DataFrame(rows)

In [ ]:
spacy_results = pd.concat([analyze_group(fp_sample, 'FP'), analyze_group(tn_sample, 'TN')], ignore_index=True)
spacy_results.to_csv(RESULTS_DIR / 'spacy_scenario2.csv', index=False, encoding='utf-8-sig')

print(spacy_results.groupby('group').mean(numeric_only=True))

## 9. Avaliação dos 4 cenários

In [ ]:
models = {'scenario1': model_s1, 'scenario3': load_scenario_model('scenario3')}

scenarios_eval = {
  'scenario1': ('scenario1', 'human'),
  'scenario2': ('scenario1', 'synthetic'),
  'scenario3': ('scenario3', 'synthetic'),
  'scenario4': ('scenario3', 'human'),
}

results = {}
for name, (model_name, fake_origin) in scenarios_eval.items():
  df_t = filter_test_data(df_news, fake_origin=fake_origin).reset_index(drop=True)
  y_true = df_t['label_id'].tolist()
  y_pred = make_predict_proba(models[model_name])(df_t['text']).argmax(axis=1)

  tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
  precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1)

  results[name] = {
    'eval_accuracy': accuracy_score(y_true, y_pred), 'eval_precision': precision,
    'eval_recall': recall, 'eval_f1': f1,
    'eval_tn': int(tn), 'eval_fp': int(fp), 'eval_fn': int(fn), 'eval_tp': int(tp)
  }

  print(name, results[name])

with open(RESULTS_DIR / 'scenario_results.json', 'w') as f:
  json.dump(results, f, indent=2, default=float)

## 10. Teste qui-quadrado

In [ ]:
def correct_incorrect(r):
  total = r['eval_tn'] + r['eval_fp'] + r['eval_fn'] + r['eval_tp']
  wrong = r['eval_fp'] + r['eval_fn']

  return [total - wrong, wrong]

In [ ]:
for a, b in [('scenario1', 'scenario2'), ('scenario2', 'scenario3'), ('scenario1', 'scenario4')]:
  chi2, p, _, _ = chi2_contingency([correct_incorrect(results[a]), correct_incorrect(results[b])])
  print(f'{a} vs {b}: chi2 = {chi2:.2f}, p = {p:.4g}')

## 11. Análise causal da queda de desempenho

In [ ]:
FEATURES = ['total_tokens', 'avg_sentence_length', 'propn_ratio', 'per_ratio', 'org_ratio', 'adj_ratio', 'adv_ratio', 'emotiveness', 'stop_ratio', 'weekday_ratio']

feat_path = RESULTS_DIR / 'features_3_classes.csv'
comp_path = RESULTS_DIR / 'comparacao_3_classes.csv'

if feat_path.exists():
  feat = pd.read_csv(feat_path, encoding='utf-8-sig')
  print(f'features_3_classes.csv carregado de {feat_path} ({len(feat)} linhas)')
else:
  test = df_news[df_news['split'] == 'test']
  df_all = pd.concat([
    test[test['label'] == 'true'].assign(classe='verdadeira'),
    test[(test['label'] == 'fake') & (test['origin'] == 'human')].assign(classe='falsa_humana'),
    test[(test['label'] == 'fake') & (test['origin'] == 'synthetic')].assign(classe='falsa_sintetica'),
  ], ignore_index=True)

  def extra_features(text):
    f = extract_linguistic_features(text)

    if f is None:
      return None

    words = re.findall(r'\w+', text.lower())
    f['stop_ratio'] = sum(w in STOP for w in words) / max(len(words), 1)
    f['weekday_ratio'] = len(WEEKDAYS.findall(text)) / f['total_tokens']

    return f

  rows = []
  for _, r in df_all.iterrows():
    f = extra_features(r['text'])

    if f:
      f['id'] = r['id']
      f['classe'] = r['classe']
      rows.append(f)

  feat = pd.DataFrame(rows)
  feat.to_csv(feat_path, index=False, encoding='utf-8-sig')

medianas = feat.groupby('classe')[FEATURES].median().T
medianas['sintetica_mais_perto_da_verdadeira'] = ((medianas['falsa_sintetica'] - medianas['verdadeira']).abs() < (medianas['falsa_humana'] - medianas['verdadeira']).abs())

print(medianas.round(4))
print('\nCaracterísticas em que a sintética está mais perto da verdadeira:', int(medianas['sintetica_mais_perto_da_verdadeira'].sum()), 'de', len(FEATURES))

pairs = [('falsa_sintetica', 'falsa_humana'), ('falsa_sintetica', 'verdadeira'), ('falsa_humana', 'verdadeira')]
rows = []

for feature in FEATURES:
  for a, b in pairs:
    xa = feat.loc[feat['classe'] == a, feature]
    xb = feat.loc[feat['classe'] == b, feature]
    u, p = mannwhitneyu(xa, xb, alternative='two-sided')
    rows.append({'feature': feature, 'A': a, 'B': b, 'AUC': u / (len(xa) * len(xb)), 'p': p})

comp = pd.DataFrame(rows)
comp['p_bonferroni'] = (comp['p'] * len(comp)).clip(upper=1)

print(comp.sort_values('p_bonferroni').round(4).to_string(index=False))
comp.to_csv(comp_path, index=False, encoding='utf-8-sig')

In [ ]:
synth_pred = df_test_s2[df_test_s2['label_id'] == 0][['id', 'predicted_label_id']]
err = synth_pred.merge(feat[feat['classe'] == 'falsa_sintetica'], on='id')
print('Falsas sintéticas analisadas:', len(err), '| enganaram o modelo:', int(err['predicted_label_id'].sum()))

X = err[['avg_sentence_length', 'propn_ratio', 'per_ratio', 'emotiveness', 'stop_ratio', 'weekday_ratio']].copy()
X['log_tokens'] = np.log(err['total_tokens'])
X = (X - X.mean()) / X.std().replace(0, 1)
y = err['predicted_label_id']

modelo_logit = sm.Logit(y, sm.add_constant(X)).fit(disp=False)
ci = modelo_logit.conf_int()

tabela = pd.DataFrame({
    'odds_ratio': np.exp(modelo_logit.params),
    'IC95_inf': np.exp(ci[0]),
    'IC95_sup': np.exp(ci[1]),
    'p': modelo_logit.pvalues,
})

print(tabela.round(4))
tabela.to_csv(RESULTS_DIR / 'regressao_erro.csv', encoding='utf-8-sig')

In [ ]:
lime_fp = pd.read_csv(RESULTS_DIR / 'lime_scenario2_fp.csv', encoding='utf-8-sig')
fp_ids = sorted(lime_fp['id'].unique())
fps = false_positives[false_positives['id'].isin(fp_ids)].reset_index(drop=True)

print(f'Falsos positivos usados (mesmos do LIME salvo): {len(fps)}')

p_orig = predict_proba(fps['text'].tolist())[:, 1]

print('Checagem (todos com P(verdadeira) > 0,5):', bool((p_orig > 0.5).all()))

ESTRUTURAIS = re.compile(r'\b(?:mas|de|que|com)\b', re.IGNORECASE)

def remove_spans(text, spans):
  out, last = [], 0

  for ini, fim in sorted(spans):
    if ini < last:
      continue
    out.append(text[last:ini])
    last = fim

  out.append(text[last:])

  return re.sub(r'\s+', ' ', ''.join(out)).strip()

def spans_entidades(text):
  return [(e.start_char, e.end_char) for e in nlp(text).ents if e.label_ in ('PER', 'ORG')]

def spans_regex(pattern):
  return lambda text: [(m.start(), m.end()) for m in pattern.finditer(text)]

def n_palavras(text, spans):
  return sum(len(text[i:f].split()) for i, f in spans)

def remove_random_words(text, n, rng):
  words = text.split()
  if n <= 0 or n >= len(words):
    return text

  drop = set(rng.choice(len(words), size=n, replace=False).tolist())

  return ' '.join(w for i, w in enumerate(words) if i not in drop)

CONDICOES = {
  'entidades PER/ORG': spans_entidades,
  'dias da semana': spans_regex(WEEKDAYS),
  'palavras estruturais (mas/de/que/com)': spans_regex(ESTRUTURAIS),
}

rng = np.random.default_rng(SEED)
resumo = []
for nome, fn in CONDICOES.items():
  idx, t_abl, t_ctrl = [], [], []
  for i, t in enumerate(fps['text']):
    spans = fn(t)
    n = n_palavras(t, spans)

    if n == 0:
      continue

    idx.append(i)
    t_abl.append(remove_spans(t, spans))
    t_ctrl.append(remove_random_words(t, n, rng))

  p0 = p_orig[idx]
  p_abl = predict_proba(t_abl)[:, 1]
  p_ctrl = predict_proba(t_ctrl)[:, 1]

  try:
    p_wilcoxon = wilcoxon(p_abl, p_ctrl).pvalue
  except ValueError:
    p_wilcoxon = float('nan')

  resumo.append({
    'caracteristica_removida': nome,
    'n_textos': len(idx),
    'P(verdadeira)_original': p0.mean(),
    'P(verdadeira)_sem_caracteristica': p_abl.mean(),
    'P(verdadeira)_controle_aleatorio': p_ctrl.mean(),
    'taxa_virou_falsa_ablacao': (p_abl < 0.5).mean(),
    'taxa_virou_falsa_controle': (p_ctrl < 0.5).mean(),
    'p_wilcoxon_ablacao_vs_controle': p_wilcoxon,
  })

resumo = pd.DataFrame(resumo)
print(resumo.round(4).T)
resumo.to_csv(RESULTS_DIR / 'intervencao_ablacao.csv', index=False, encoding='utf-8-sig')